In [13]:
import requests
import math
import pandas as pd
import json

url = "https://apis.data.go.kr/B551982/rte/rtm_loc_info"

params = {
    "serviceKey": "6299fdd61b86d0676ec07175a4dc3d87817a46689120fa3ffe3588e2ae34bdff",
    "pageNo": 1,
    "numOfRows": 100,
    "stdgCd": "4686000000"
}

In [ ]:
!pip install pandas

In [14]:
print(params["serviceKey"])
print(len(params["serviceKey"]))

6299fdd61b86d0676ec07175a4dc3d87817a46689120fa3ffe3588e2ae34bdff
64


In [15]:
response = requests.get(url, params=params)

print("상태코드:", response.status_code)
print("응답 앞부분:")
print(response.text[:500])

상태코드: 200
응답 앞부분:
{"header":{"resultCode":"K0","resultMsg":"NORMAL_SERVICE","context":"https://ido.sharedata.go.kr/dcat/catalog/c0006.json","dataset":"KLID-DATASET-d0006","service":"KLID-d0006-s0003"},"body":{"totalCount":800,"pageNo":1,"numOfRows":100,"items":{"item":[{"stdgCd":"4686000000","lclgvNm":"전라남도 함평군","rteId":"10","vhclNo":"251016","gthrDt":"2026-04-04 00:00:00","rteNo":"100","lat":"35.0621109000","lot":"126.5234756500","oprDrct":"52","oprSpd":"49","evtCd":"","evtType":"GPS","totDt":"20260404081901"},{


In [16]:
response = requests.get(url, params=params)
data = response.json()

total_count = data["body"]["totalCount"]
num_of_rows = data["body"]["numOfRows"]
total_pages = math.ceil(total_count / num_of_rows)

all_items = data["body"]["items"]["item"]

for page in range(2, total_pages + 1):
    params["pageNo"] = page
    response = requests.get(url, params=params)
    data = response.json()
    all_items.extend(data["body"]["items"]["item"])

df = pd.DataFrame(all_items)
print(df.head())
print("총 데이터 개수:", len(df))

       stdgCd   lclgvNm rteId  vhclNo               gthrDt rteNo  \
0  4686000000  전라남도 함평군    10  251016  2026-04-04 00:00:00   100   
1  4686000000  전라남도 함평군    10  251032  2026-04-01 00:00:00   100   
2  4686000000  전라남도 함평군    10  252000  2026-03-31 00:00:00   100   
3  4686000000  전라남도 함평군    10  252001  2026-04-03 00:00:00   100   
4  4686000000  전라남도 함평군    10  252002  2026-03-30 00:00:00   100   

             lat             lot oprDrct oprSpd evtCd evtType           totDt  
0  35.0621109000  126.5234756500      52     49           GPS  20260404081901  
1  35.0622253400  126.5232162500     315     31           GPS  20260401110901  
2  35.0629539500  126.5225906400     325     23           GPS  20260331111200  
3  35.0609817500  126.5237503100     338     47           GPS  20260403081902  
4  35.0626411400  126.5223617600      85     32           GPS  20260330110901  
총 데이터 개수: 800


In [17]:
print(df.info())
print(df.describe(include="all"))
print(df["rteNo"].value_counts().head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 13 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   stdgCd   800 non-null    object
 1   lclgvNm  800 non-null    object
 2   rteId    800 non-null    object
 3   vhclNo   800 non-null    object
 4   gthrDt   800 non-null    object
 5   rteNo    800 non-null    object
 6   lat      800 non-null    object
 7   lot      800 non-null    object
 8   oprDrct  800 non-null    object
 9   oprSpd   800 non-null    object
 10  evtCd    800 non-null    object
 11  evtType  800 non-null    object
 12  totDt    800 non-null    object
dtypes: object(13)
memory usage: 81.4+ KB
None
            stdgCd   lclgvNm rteId  vhclNo               gthrDt rteNo    lat  \
count          800       800   800     800                  800   800    800   
unique           1         1    71      36                   28     2    471   
top     4686000000  전라남도 함평군    44  251038  2026-04-0

In [18]:
df[["lclgvNm", "rteNo", "vhclNo", "lat", "lot", "oprSpd"]].head()

,lclgvNm,rteNo,vhclNo,lat,lot,oprSpd
0,전라남도 함평군,100,251016,35.0621109000,126.5234756500,49
1,전라남도 함평군,100,251032,35.0622253400,126.5232162500,31
2,전라남도 함평군,100,252000,35.0629539500,126.5225906400,23
3,전라남도 함평군,100,252001,35.0609817500,126.5237503100,47
4,전라남도 함평군,100,252002,35.0626411400,126.5223617600,32


In [19]:
df[df["oprSpd"].astype(int) >= 40][["rteNo", "vhclNo", "oprSpd"]].head()

,rteNo,vhclNo,oprSpd
0,100,251016,49
3,100,252001,47
5,100,252005,47
6,100,252006,42
16,500,251045,42


In [20]:
df["rteNo"].value_counts()

rteNo
500    570
100    230
Name: count, dtype: int64

In [21]:
df.to_csv("hampyeong_bus.csv", index=False, encoding="utf-8-sig")

In [22]:
print(df["rteNo"].value_counts())
print(df["evtType"].value_counts())

rteNo
500    570
100    230
Name: count, dtype: int64
evtType
GPS    638
       162
Name: count, dtype: int64


In [23]:
df["oprSpd"] = pd.to_numeric(df["oprSpd"], errors="coerce")
print(df["oprSpd"].describe())

count    800.000000
mean      27.643750
std       24.410124
min        0.000000
25%        0.000000
50%       28.000000
75%       49.250000
max       88.000000
Name: oprSpd, dtype: float64


In [24]:
df["totDt"] = pd.to_datetime(df["totDt"], format="%Y%m%d%H%M%S")
print(df.sort_values("totDt", ascending=False).head())

         stdgCd   lclgvNm rteId  vhclNo               gthrDt rteNo  \
690  4686000000  전라남도 함평군    81  251002  2025-10-17 00:00:00   500   
210  4686000000  전라남도 함평군    25  251032  2026-04-06 00:00:00   100   
25   4686000000  전라남도 함평군   101  251021  2026-04-06 00:00:00   500   
757  4686000000  전라남도 함평군    92  252002  2026-04-06 00:00:00   100   
688  4686000000  전라남도 함평군    81  251000  2026-04-06 00:00:00   500   

               lat             lot oprDrct  oprSpd evtCd evtType  \
690  35.1701049800  126.6646652200     309      49           GPS   
210  35.0620575000  126.5237960800     169      39           GPS   
25   35.1495208700  126.7689132700     260      31           GPS   
757  35.0605812100  126.5239181500     158      61           GPS   
688  35.0679512000  126.5347137500     107      29           GPS   

                  totDt  
690 2026-04-06 16:37:01  
210 2026-04-06 16:37:01  
25  2026-04-06 16:37:01  
757 2026-04-06 16:37:01  
688 2026-04-06 16:37:01  


In [25]:
df.to_csv("hampyeong_bus.csv", index=False, encoding="utf-8-sig")

In [31]:
df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
df["lot"] = pd.to_numeric(df["lot"], errors="coerce")

print(df[(df["lat"] < 30) | (df["lat"] > 40) | (df["lot"] < 120) | (df["lot"] > 130)][["rteNo", "vhclNo", "lat", "lot"]])

    rteNo  vhclNo  lat  lot
49    500  251029  0.0  0.0
99    500  251025  0.0  0.0
188   100  252002  0.0  0.0
198   100  252002  0.0  0.0
237   100  252002  0.0  0.0
247   100  252002  0.0  0.0
257   100  252002  0.0  0.0
269   100  252002  0.0  0.0
273   100  252002  0.0  0.0
346   500  251029  0.0  0.0
361   500  251025  0.0  0.0
363   500  251029  0.0  0.0
380   500  251025  0.0  0.0
382   500  251029  0.0  0.0
389   500  251045  0.0  0.0
418   500  251025  0.0  0.0
451   500  251025  0.0  0.0
533   500  251029  0.0  0.0
585   500  251019  0.0  0.0
589   500  251025  0.0  0.0
604   500  251025  0.0  0.0
700   500  251025  0.0  0.0
719   500  251029  0.0  0.0
732   500  251025  0.0  0.0
738   500  251000  0.0  0.0
751   100  252002  0.0  0.0
783   100  252002  0.0  0.0
790   500  251025  0.0  0.0


In [ ]:
!pip install folium

In [30]:
import folium
import pandas as pd

df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
df["lot"] = pd.to_numeric(df["lot"], errors="coerce")
df["oprSpd"] = pd.to_numeric(df["oprSpd"], errors="coerce")

map_df = df.dropna(subset=["lat", "lot"]).copy()

center_lat = map_df["lat"].mean()
center_lot = map_df["lot"].mean()

m = folium.Map(location=[center_lat, center_lot], zoom_start=13)

color_dict = {
    "100": "blue",
    "500": "red"
}

for _, row in map_df.iterrows():
    route = str(row["rteNo"])
    color = color_dict.get(route, "gray")
    
    popup_text = f"""
    지역: {row['lclgvNm']}<br>
    노선번호: {row['rteNo']}<br>
    차량번호: {row['vhclNo']}<br>
    속도: {row['oprSpd']} km/h<br>
    수집시각: {row['totDt']}
    """
    
    folium.CircleMarker(
        location=[row["lat"], row["lot"]],
        radius=5,
        popup=folium.Popup(popup_text, max_width=250),
        fill=True,
        color=color,
        fill_color=color,
        fill_opacity=0.7
    ).add_to(m)

m.save("hampyeong_bus_map.html")
import webbrowser
webbrowser.open("hampyeong_bus_map.html")

True